# Visualizing the doop benchmark

Builds the doop points-to analysis program (78 rules, 74 relations,
96 runner variants once recursive rules expand for semi-naive eval)
and renders it as an inline iframe in each cell. The renderer is the
same React/reactflow webview as the [srdatalog-viz VS Code extension](https://github.com/harp-lab/srdatalog-viz).

**Setup:** install the project editable + Jupyter test deps:
```bash
uv sync --extra test-jupyter --group dev
```
Then in VS Code: "Select Kernel" → `.venv/bin/python`.

**If you don't see a graph below**, restart the kernel — earlier kernel
sessions cached an older `_repr_mimebundle_` that didn't emit
`text/html`. Then re-run cells in order.

## 1. Build the program

`doop.py` is auto-translated from the Nim source. The builder takes
a `meta` dict — those are the dataset-resolved integer constants
(class IDs, method IDs) pre-interned by `prepare_vpt_dataset.dl`.
Different inputs (batik, biojava, eclipse) get different metas;
the rule structure is identical across them.

In [ ]:
import json
from pathlib import Path

import doop

META_PATH = Path('../../SRDatalog/integration_tests/examples/doop/batik_meta.json')
meta = json.loads(META_PATH.read_text())
prog = doop.build_doopdb_program(meta)
len(prog.rules), len(prog.relations)

## 2. Ruleset overview (default — dark)

Leaving `prog` as the last expression of a cell triggers
`Program._repr_mimebundle_`, which emits a `text/html` iframe
showing the rule dependency graph for the whole program. Stratum
groupings are color-coded; recursive strata get a distinct ring.

In [ ]:
prog

## 3. Light mode

`prog.show(theme='light')` — same view, light palette. Useful for
screenshots / PDFs / VS Code's light theme. Other valid values:
`'dark'` (default), `'high-contrast'`.

In [ ]:
prog.show(theme='light')

## 4. Per-rule plan view

`prog.show(rule='VPT_LoadField')` drills into one rule. The renderer
switches to its plan view — shows variants for each delta seed (in
recursive rules), access patterns per body atom, current
`clauseOrder` and `varOrder`. Drag the items in the sidebar to
reorder; the graph re-lays out live (the change is local to the
iframe; round-tripping back to the .py source is the labextension
follow-up work).

In [ ]:
prog.show(rule='VPT_LoadField')

## 5. One specific version of a rule (delta filter)

A recursive rule with N body references to recursive relations
expands into N variants under semi-naive evaluation — one per
delta seed. The renderer's plan view shows them all in a tab
switcher. To **isolate one specific version**, pass `delta=K`
alongside `rule=...` — only the variant with `deltaIdx == K`
renders, useful when you want to inspect one access-pattern
set in isolation without the others' clutter.

Example below: doop's `VPT_LoadField` rule has multiple
delta variants because three of its body atoms reference recursive
relations (`VarPointsTo`, `FieldPointsTo`, `AssignableInst`)
— so semi-naive emits three variants, with deltaIdx ∈ {0, 1, 2}.

In [ ]:
# How many delta variants does this rule have? Walk the bundle.
from srdatalog.viz.bundle import get_visualization_bundle

bundle = get_visualization_bundle(prog)
for s in bundle['hir']['strata']:
  for v in s.get('recursive', []):
    if v['rule']['name'] == 'VPT_LoadField':
      print(
        f"deltaIdx={v.get('deltaIdx')}  body[deltaIdx]={v['rule']['body'][v['deltaIdx']]['rel']}"
      )

In [ ]:
# Render just delta=0 — the variant seeded on body clause 0.
prog.show(rule='VPT_LoadField', delta=0)

In [ ]:
# Render just delta=1 — the variant seeded on body clause 1.
# Compare access patterns side-by-side with the cell above.
prog.show(rule='VPT_LoadField', delta=1)

## 6. Combined options

All of `rule`, `theme`, `include_jit`, `height_px` compose. For
performance debugging — pick a single rule + light mode + tall
iframe + JIT C++ kernel inspector enabled:

In [ ]:
prog.show(rule='VPT_HeapAlloc', theme='light', include_jit=True, height_px=900)

## 7. Drill via Python directly

If you want to inspect the data without going through the renderer:

In [ ]:
from srdatalog.viz.bundle import get_visualization_bundle

bundle = get_visualization_bundle(prog, include_jit=True)
[r for r in bundle['rules'] if 'VarPointsTo' in r['heads']][:3]

In [ ]:
# Generated kernel C++ for one runner — first 1500 chars
name = next(iter(bundle['jit']))
print(f'=== {name} ===')
print(bundle['jit'][name][:1500])